<a href="https://colab.research.google.com/github/OlhaZahrebelna/Recommendation-Systems-Goodbooks-10k/blob/main/RecSys_Goodbooks_Solution_Eng%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recommender Systems on Real-World Data: Goodbooks-10k

## Project overview

This learning project explores several recommender-system architectures using the **Goodbooks-10k** dataset. The objective is to compare simple content-based retrieval with neural retrieval and ranking approaches on realistic, sparse user–item interaction data.

The project covers:

- feature engineering from noisy user-generated tags;
- a Vector Space Model;
- a Two-Tower retrieval model;
- a concat-based Neural Collaborative Filtering ranker;
- a two-stage Retrieval → Ranking pipeline;
- offline evaluation with Recall@K;
- limitations, cold-start behavior, diversity, and possible improvements.

**Technology stack:** `NumPy`, `pandas`, and `PyTorch`.

A GPU is optional, although it can reduce training time in Google Colab.

---

## Dataset

[Goodbooks-10k](https://www.kaggle.com/datasets/zygmunt/goodbooks-10k) contains approximately 6 million ratings for 10,000 popular books from 53,424 users.

Main files:

- `ratings.csv` — `user_id`, `book_id`, and rating values from 1 to 5;
- `books.csv` — book metadata such as title, author, publication year, and average rating;
- `book_tags.csv` — user-generated tags assigned to books;
- `tags.csv` — mapping between tag IDs and tag names.

The dataset does not provide a clean genre column. Genres must therefore be derived from user-generated tags, which creates a realistic feature-engineering problem.

Another practical issue is that `book_tags.csv` uses `goodreads_book_id`, while `ratings.csv` uses `book_id`. The two tables must be connected through `books.csv`.


## 1. Data loading

In [1]:
import os
import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/zygmuntz/goodbooks-10k/master"
FILES = ["ratings.csv", "books.csv", "book_tags.csv", "tags.csv"]

def load(fname):
    """Load a local file first; otherwise use the GitHub mirror."""
    if os.path.exists(fname):
        return pd.read_csv(fname)
    print(f"{fname} was not found locally — loading it from GitHub...")
    return pd.read_csv(f"{GITHUB}/{fname}")

ratings = load("ratings.csv")
books = load("books.csv")
book_tags = load("book_tags.csv")
tags = load("tags.csv")

print("ratings:", ratings.shape)
print("books:  ", books.shape)
print("book_tags:", book_tags.shape, "| tags:", tags.shape)

ratings.csv was not found locally — loading it from GitHub...
books.csv was not found locally — loading it from GitHub...
book_tags.csv was not found locally — loading it from GitHub...
tags.csv was not found locally — loading it from GitHub...
ratings: (5976479, 3)
books:   (10000, 23)
book_tags: (999912, 3) | tags: (34252, 2)


In [2]:
books[["book_id", "authors", "title", "average_rating"]].head()

,book_id,authors,title,average_rating
0,1,Suzanne Collins,"The Hunger Games (The Hunger Games, #1)",4.34
1,2,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,4.44
2,3,Stephenie Meyer,"Twilight (Twilight, #1)",3.57
3,4,Harper Lee,To Kill a Mockingbird,4.25
4,5,F. Scott Fitzgerald,The Great Gatsby,3.89


## 2. Genre feature engineering

The dataset does not include a structured genre field. Instead, it contains noisy user-generated tags.

This section selects a set of canonical genres and converts the available tags into a binary **book × genre** matrix. Each row represents a book, and each column indicates whether users associated the book with a given genre.

These binary genre indicators become the content features used by the recommender models.

In [3]:
GENRES = ["fantasy", "romance", "mystery", "thriller", "horror", "historical",
          "science-fiction", "young-adult", "nonfiction", "classics",
          "contemporary", "crime"]

name_to_tagid = dict(zip(tags["tag_name"], tags["tag_id"]))
genre_tag_ids = {g: name_to_tagid[g] for g in GENRES if g in name_to_tagid}

gid_to_bid = dict(zip(books["goodreads_book_id"], books["book_id"]))
tagid_to_genre = {tid: g for g, tid in genre_tag_ids.items()}

bt = book_tags[book_tags["tag_id"].isin(genre_tag_ids.values())].copy()
bt["book_id"] = bt["goodreads_book_id"].map(gid_to_bid)
bt = bt.dropna(subset=["book_id"])
bt["genre"] = bt["tag_id"].map(tagid_to_genre)


genre_matrix = (
    bt.pivot_table(index="book_id", columns="genre", values="count", aggfunc="sum", fill_value=0)
      .reindex(columns=GENRES, fill_value=0) > 0
).astype(int)

print("A book in at least one genre:", (genre_matrix.sum(axis=1) > 0).sum(), "/", len(books))
print("\nGenre distribution:")
print(genre_matrix.sum().sort_values(ascending=False))
genre_matrix.head()

A book in at least one genre: 9954 / 10000

Genre distribution:
genre
contemporary       5287
fantasy            4259
romance            4251
mystery            3686
young-adult        3630
classics           2785
historical         2544
thriller           2522
science-fiction    2222
crime              2083
nonfiction         1833
horror             1372
dtype: int64


genre,fantasy,romance,mystery,thriller,horror,historical,science-fiction,young-adult,nonfiction,classics,contemporary,crime
book_id,,,,,,,,,,,,
1,1,1,0,1,0,0,1,1,0,0,1,0
2,1,0,1,0,0,0,0,1,0,1,1,0
3,1,0,0,0,1,0,1,1,0,0,1,0
4,0,0,1,0,0,1,0,1,0,1,1,1
5,0,1,0,0,0,1,0,1,0,1,0,0


## 3. Data sampling

The complete dataset contains roughly 6 million ratings, which is unnecessarily large for an educational CPU-based notebook.

To keep training practical while preserving meaningful interaction density, the project:

- keeps the most popular books;
- keeps users with at least 20 ratings;
- samples a fixed number of active users;
- removes books without genre features.

The sampling parameters can be increased when more compute is available.


In [4]:
TOP_BOOKS = 1500
MIN_USER_RATINGS = 20
N_USERS = 2000
LIKE_THRESHOLD = 4

rng = np.random.RandomState(42)

top_books = ratings["book_id"].value_counts().head(TOP_BOOKS).index
r = ratings[ratings["book_id"].isin(top_books)]
active = r["user_id"].value_counts()
r = r[r["user_id"].isin(active[active >= MIN_USER_RATINGS].index)]
sample_users = rng.choice(r["user_id"].unique(), size=min(N_USERS, r["user_id"].nunique()), replace=False)
r = r[r["user_id"].isin(sample_users)].copy()

r = r[r["book_id"].isin(genre_matrix.index)].copy()

items = sorted(r["book_id"].unique())
users = sorted(r["user_id"].unique())
genre_matrix = genre_matrix.reindex(items).fillna(0).astype(int)

print(f"Interactions: {len(r):,} | users: {len(users):,} | books: {len(items):,}")
print(f"Density: {len(r) / (len(users) * len(items)):.4f}")

Interactions: 140,934 | users: 2,000 | books: 1,496
Density: 0.0471


In [5]:
import torch
import torch.nn as nn

torch.manual_seed(42)

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {b: i for i, b in enumerate(items)}
title_of = dict(zip(books["book_id"], books["title"]))

item_feats = torch.tensor(genre_matrix.values, dtype=torch.float32)
M = len(items)
n_genres = item_feats.shape[1]

r = r.sample(frac=1, random_state=42).reset_index(drop=True)
n_val = int(len(r) * 0.2)
val_df = r.iloc[:n_val]
train_df = r.iloc[n_val:]

train_pos = train_df[train_df["rating"] >= LIKE_THRESHOLD]
pos_u = torch.tensor([user_to_idx[u] for u in train_pos["user_id"]])
pos_i = torch.tensor([item_to_idx[b] for b in train_pos["book_id"]])


from collections import defaultdict
seen_by_user = defaultdict(set)
for u, b in zip(train_df["user_id"], train_df["book_id"]):
    seen_by_user[user_to_idx[u]].add(item_to_idx[b])

val_pos = defaultdict(set)
for row in val_df.itertuples():
    if row.rating >= LIKE_THRESHOLD:
        val_pos[user_to_idx[row.user_id]].add(item_to_idx[row.book_id])

print(f"Positive training pairs: {len(pos_u):,} | users with validation positives: {len(val_pos):,}")

Positive training pairs: 77,070 | users with validation positives: 1,991


## 4. Ranking evaluation with Recall@K

Rating-prediction metrics such as RMSE measure how accurately a model predicts explicit rating values. In a recommendation interface, however, the main objective is usually to place relevant items near the top of a ranked list.

This project therefore uses **Recall@K**.

For each user, Recall@K measures how many relevant validation items appear among the top-K recommendations. The final metric is aggregated across users.

Other common ranking metrics include Precision@K, NDCG, MAP, and MRR.

In [6]:
def recall_at_k(score_fn, k=10):
    """Return the share of validation positives found in the top-K recommendations.
    score_fn(user_idx_tensor) -> score matrix with shape (n_users, M)."""
    eval_users = list(val_pos.keys())
    hits, total = 0, 0
    with torch.no_grad():
        scores = score_fn(torch.tensor(eval_users))
        for row, u in enumerate(eval_users):
            s = scores[row].clone()
            for i in seen_by_user[u]:
                s[i] = -1e9
            topk = torch.topk(s, k).indices.tolist()
            truth = val_pos[u]
            hits += len(set(topk) & truth)
            total += min(len(truth), k)
    return hits / max(total, 1)

## 5. Baseline: Vector Space Model

The first model represents books and users in the same genre-feature space.

Book representations are L2-normalized genre vectors. A user representation is calculated as the weighted average of the genre vectors of books the user liked. Recommendations are ranked with cosine similarity.

This model provides a simple, interpretable content-based baseline.

In [7]:
item_emb = torch.nn.functional.normalize(item_feats, dim=1)


def user_vector(user_idx):
    liked = [i for i in seen_by_user[user_idx]]
    return None


user_vecs = torch.zeros(len(users), n_genres)
counts = torch.zeros(len(users), 1)
liked_ratings = torch.tensor(train_pos["rating"].values, dtype=torch.float32).unsqueeze(1)
user_vecs.index_add_(0, pos_u, item_emb[pos_i] * liked_ratings)
counts.index_add_(0, pos_u, liked_ratings)
user_vecs = user_vecs / counts.clamp(min=1e-8)
user_vecs = torch.nn.functional.normalize(user_vecs, dim=1)

def vsm_scores(user_idxs):
    return user_vecs[user_idxs] @ item_emb.T

print(f"Vector Space Model Recall@10 = {recall_at_k(vsm_scores, 10):.3f}")

u = next(iter(val_pos))
s = vsm_scores(torch.tensor([u]))[0].clone()
for i in seen_by_user[u]:
    s[i] = -1e9
recs = torch.topk(s, 5).indices.tolist()
print(f"\nTop 5 recommendations for user #{u}:")
for j in recs:
    print("  -", title_of.get(items[j], items[j]))


Vector Space Model Recall@10 = 0.052

Top 5 recommendations for user #316:
  - The Battle of the Labyrinth (Percy Jackson and the Olympians, #4)
  - Hush, Hush (Hush, Hush, #1)
  - Fallen (Fallen, #1)
  - Shadow Kiss (Vampire Academy, #3)
  - Beautiful Creatures (Caster Chronicles, #1)


### Interpretation

The Vector Space Model usually produces a relatively low Recall@10 because every book is described with only 12 broad binary genres.

Many different books receive identical or nearly identical genre vectors. As a result, the model can identify broad preferences but cannot distinguish well between books within the same genre combination.

This creates a clear performance ceiling for a content-based system with limited item features.

## 6. Two-Tower retrieval model

The Two-Tower architecture learns separate representations for users and books.

- The **user tower** starts from a trainable user embedding.
- The **item tower** transforms genre features into a dense item embedding.
- Positive user–book pairs are moved closer together.
- Randomly sampled negative pairs are moved farther apart.

Both output vectors are L2-normalized. Their dot product is used as the relevance score.

The main production advantage is retrieval efficiency: item embeddings can be precomputed and stored in an approximate-nearest-neighbor index such as FAISS.


In [8]:
EMB_DIM = 32

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_emb = nn.Embedding(len(users), EMB_DIM)
        self.user_tower = nn.Sequential(nn.Linear(EMB_DIM, 64), nn.ReLU(), nn.Linear(64, EMB_DIM))
        self.item_tower = nn.Sequential(nn.Linear(n_genres, 64), nn.ReLU(), nn.Linear(64, EMB_DIM))
    def user_vec(self, u):
        return torch.nn.functional.normalize(self.user_tower(self.user_emb(u)), dim=-1)
    def item_vec(self, feats):
        return torch.nn.functional.normalize(self.item_tower(feats), dim=-1)
    def forward(self, u, feats):
        return (self.user_vec(u) * self.item_vec(feats)).sum(-1)

model_tt = TwoTower()
opt = torch.optim.Adam(model_tt.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()
P = len(pos_u)
N_NEG = 4
TEMP = 10.0

for epoch in range(15):
    neg_u = pos_u.repeat(N_NEG)
    neg_i = torch.randint(0, M, (P * N_NEG,))   # негативи з усього корпусу
    u = torch.cat([pos_u, neg_u])
    it = torch.cat([pos_i, neg_i])
    y = torch.cat([torch.ones(P), torch.zeros(P * N_NEG)])
    logits = model_tt(u, item_feats[it]) * TEMP
    loss = loss_fn(logits, y)
    opt.zero_grad(); loss.backward(); opt.step()

print(f"Two-Tower final loss: {loss.item():.4f}")

model_tt.eval()
with torch.no_grad():
    tt_item_vecs = model_tt.item_vec(item_feats)   # це кладеться у FAISS

def tt_scores(user_idxs):
    return model_tt.user_vec(user_idxs) @ tt_item_vecs.T

print(f"Two-Tower Recall@10 = {recall_at_k(tt_scores, 10):.3f}")

u = next(iter(val_pos))
s = tt_scores(torch.tensor([u]))[0].clone()
for i in seen_by_user[u]:
    s[i] = -1e9
recs = torch.topk(s, 5).indices.tolist()
print(f"\nТоп-5 recommendation for user #{u}:")
for j in recs:
    print("  -", title_of.get(items[j], items[j]))

Two-Tower final loss: 0.6395
Two-Tower Recall@10 = 0.021

Топ-5 recommendation for user #316:
  - Where the Wild Things Are
  - The Giving Tree
  - Green Eggs and Ham
  - Alice's Adventures in Wonderland & Through the Looking-Glass
  - Charlotte's Web


### Interpretation

The Two-Tower model may perform worse than the simpler Vector Space Model in this experiment.

This does not necessarily indicate an implementation error. The item tower still receives only 12 broad genre indicators, so the neural network cannot recover information that is absent from the input.

The architecture becomes more valuable when item representations include richer signals such as full tag vectors, descriptions, title embeddings, images, author information, and substantially more interaction data.

This experiment illustrates an important principle: additional model complexity does not automatically create additional business value.


## 7. Concat-based ranking model

The next model uses early fusion.

A trainable user embedding is concatenated with the book's genre features and passed through a multilayer perceptron. The model can therefore learn nonlinear interactions between user identity and item attributes.

Unlike the Two-Tower model, this architecture cannot precompute a single reusable score representation for every book. It must evaluate individual user–item pairs, making it better suited to ranking a relatively small candidate set rather than searching the entire catalogue.


In [9]:
class NCF(nn.Module):
    def __init__(self):
        super().__init__()
        self.user_emb = nn.Embedding(len(users), EMB_DIM)
        self.mlp = nn.Sequential(
            nn.Linear(EMB_DIM + n_genres, 64), nn.ReLU(),
            nn.Linear(64, 16), nn.ReLU(),
            nn.Linear(16, 1),
        )
    def forward(self, u, feats):
        x = torch.cat([self.user_emb(u), feats], dim=-1)  # EARLY FUSION
        return self.mlp(x).squeeze(-1)

model_ncf = NCF()
opt = torch.optim.Adam(model_ncf.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(15):
    neg_u = pos_u.repeat(N_NEG)
    neg_i = torch.randint(0, M, (P * N_NEG,))
    u = torch.cat([pos_u, neg_u]); it = torch.cat([pos_i, neg_i])
    y = torch.cat([torch.ones(P), torch.zeros(P * N_NEG)])
    loss = loss_fn(model_ncf(u, item_feats[it]), y)
    opt.zero_grad(); loss.backward(); opt.step()

print(f"NCF final loss: {loss.item():.4f}")

model_ncf.eval()
def rank_ncf(user_idx, candidate_idxs):
    u = torch.tensor([user_idx] * len(candidate_idxs))
    feats = item_feats[candidate_idxs]
    with torch.no_grad():
        scores = torch.sigmoid(model_ncf(u, feats)).tolist()
    ranked = sorted(zip(candidate_idxs, scores), key=lambda x: -x[1])
    return ranked

NCF final loss: 0.5153


## 8. Two-stage Retrieval → Ranking pipeline

Large recommender systems commonly separate recommendation into two stages:

1. **Retrieval:** the Two-Tower model quickly selects a small candidate set from the full catalogue.
2. **Ranking:** the concat-based neural model scores those candidates more precisely.

This design balances latency and model complexity. Retrieval reduces the search space, while ranking applies a more expensive model only to a manageable number of candidates.

In [10]:
def retrieve(user_idx, n_candidates=50):
    """Stage 1: retrieve candidates from the full catalogue with Two-Tower."""
    with torch.no_grad():
        uv = model_tt.user_vec(torch.tensor([user_idx]))[0]
        sims = (tt_item_vecs @ uv).clone()
    for i in seen_by_user[user_idx]:
        sims[i] = -1e9
    return torch.topk(sims, n_candidates).indices.tolist()

def recommend_pipeline(user_idx, n_candidates=50, top_k=5):
    """Stage 1 retrieval followed by Stage 2 NCF ranking."""
    candidates = retrieve(user_idx, n_candidates)
    ranked = rank_ncf(user_idx, candidates)[:top_k]
    return candidates, ranked

for u in list(val_pos)[:3]:
    cands, ranked = recommend_pipeline(u, n_candidates=50, top_k=5)
    print(f"User #{u}")
    print(f"  retrieval -> {len(cands)} candidates (top 3: {[title_of.get(items[c]) for c in cands[:3]]})")
    print(f"  ranking   -> top-5:")
    for j, sc in ranked:
        print(f"      {sc:.3f}  {title_of.get(items[j], items[j])}")
    print()

User #316
  retrieval -> 50 candidates (top 3: ['Peter Pan', 'Alice in Wonderland', 'The Last Battle (Chronicles of Narnia, #7)'])
  ranking   -> top-5:
      0.223  Horton Hears a Who!
      0.223  Cloudy With a Chance of Meatballs
      0.223  Peter Pan
      0.223  Alice in Wonderland
      0.223  The Last Battle (Chronicles of Narnia, #7)

User #1063
  retrieval -> 50 candidates (top 3: ['Alexander and the Terrible, Horrible, No Good, Very Bad Day', 'Madeline', 'Amelia Bedelia  (Amelia Bedelia #1)'])
  ranking   -> top-5:
      0.324  Alexander and the Terrible, Horrible, No Good, Very Bad Day
      0.324  Madeline
      0.324  Amelia Bedelia  (Amelia Bedelia #1)
      0.324  Charlie and the Chocolate Factory (Charlie Bucket, #1)
      0.324  Love You Forever

User #788
  retrieval -> 50 candidates (top 3: ['Oliver Twist', 'How the Grinch Stole Christmas!', 'The Polar Express'])
  ranking   -> top-5:
      0.256  Oliver Twist
      0.249  How the Grinch Stole Christmas!
      0.249

### Why not rank the entire catalogue directly?

A production catalogue may contain millions of items. Running a relatively expensive pairwise ranking model for every user–item combination would create unacceptable latency and infrastructure cost.

A lightweight retrieval stage removes most clearly irrelevant items. The ranker then evaluates only the remaining candidates.

## 9. Project analysis

The final section summarizes the main findings and discusses the limitations of the current implementation.

Topics include:

1. reasons for low Recall@10;
2. feature improvements without changing the model architecture;
3. recommendation diversity;
4. item cold start;
5. the difference between optimizing clicks and optimizing long-term engagement.


## 10. Conclusions and improvement opportunities

### 1. Why is Recall@10 relatively low?

The models use only 12 binary genre features. Thousands of different books therefore receive identical or nearly identical representations, which makes fine-grained ranking difficult.

The interaction matrix is also highly sparse. Even active users have rated only a small part of the catalogue.

Offline evaluation introduces another limitation: only books rated positively in the validation set are treated as relevant. A user may like many other unseen books, but the metric still counts those recommendations as misses because the dataset does not contain complete preference labels.

### 2. How could quality improve without changing the architecture?

The existing models could use richer inputs:

- author embeddings or one-hot features;
- publication year;
- average rating and rating count;
- the complete tag vocabulary represented with TF-IDF;
- text embeddings generated from titles or descriptions;
- aggregated user-history statistics.

Richer features would allow both simple and neural models to learn more informative distinctions between books.

### 3. Diversity

A recommendation list containing ten very similar fantasy books may be individually relevant but still feel repetitive.

Diversity can be introduced through re-ranking. One option is **Maximal Marginal Relevance**, which balances predicted relevance against similarity to items already selected. Another option is to apply explicit constraints, such as limiting the number of books from the same genre or author.

### 4. Freshness and item cold start

The Vector Space Model and the content-based item tower can score a new book immediately when genre or other content features are available.

A purely collaborative model cannot learn a useful representation for a new item with no interactions. This is the item cold-start problem and is one reason production recommender systems often combine collaborative and content-based signals.

### 5. Engagement objectives

Click-through rate can reward clickbait: a user may click an item but leave immediately.

Watch time, reading time, or another engagement-duration signal can better reflect whether the content delivered value. In weighted logistic regression, positive examples can receive weights proportional to engagement duration. The model then places more emphasis on interactions that generated sustained engagement rather than only a click.

---

## Final takeaway

The project demonstrates a complete recommendation workflow:

- preparing interaction and content data;
- engineering book features;
- building retrieval and ranking models;
- evaluating top-K recommendations;
- combining models in a two-stage pipeline;
- identifying practical limitations and improvement paths.

The strongest lesson is that model architecture alone is not enough. Recommendation quality depends heavily on feature richness, data coverage, negative sampling, evaluation design, and serving constraints.
